# MovieLens 25M — Phase 2: Standalone Models & Validation Benchmarks

**Goal.** Implement and fairly compare four standalone recommenders under the frozen Phase 1 protocol, then record validation benchmarks, tuning evidence and resource requirements.

| Task | What it covers |
|---|---|
| P2.1 | Popularity baseline (ranking) |
| P2.2 | Regularized bias baseline (rating) |
| P2.3 | Item-based CF (rating) |
| P2.4 | Matrix factorization / ALS (rating) |
| P2.5 | Comparable tuning budgets and selection |
| P2.6 | Common recommendation lists (shared eligibility and ranking) |
| P2.7 | Standalone metrics (NDCG@10, Recall@10, coverage, exposure, RMSE, MAE) |
| P2.8 | Runtime and memory requirements |
| P2.9 | Peer review / independent reproduction |

Each step below explains the idea and the formula, then runs the real code in `src/` so results are verifiable end to end.

## 0. How to use this notebook

- Run `python -m src.prepare_datasets` once so `data/processed/` exists.
- Select the project `.venv` kernel (needs numpy, pandas, scipy, psutil, scikit-learn).
- Run cells top to bottom; later cells reuse `history`, `catalogue`, `positives`, `cohort`, `fallback`.
- Item-CF and MF fits take a few minutes each.

In [1]:
import os
import sys
import platform
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import psutil

# Locate the project root (the folder that contains src/) and run from there.
root = Path.cwd()
while root != root.parent and not (root / 'src').is_dir():
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

PROCESS = psutil.Process()

def rss_mb():
    """Resident set size of this process in MB."""
    return PROCESS.memory_info().rss / (1024 ** 2)

pd.set_option('display.width', 160)
print('Project root :', root)
print('Python       :', platform.python_version())
print('NumPy/Pandas :', np.__version__, '/', pd.__version__)

Project root : d:\SUTD\statistical\projects\statistical-learning-movielens
Python       : 3.13.15
NumPy/Pandas : 2.5.4 / 3.1.1


## 1. Evaluation protocol (frozen in Phase 1)

All models share one protocol.

- **History** $H$: every rating strictly before `2017-01-01` UTC ($|H| = 20{,}798{,}765$).
- **Outcomes** $O$: 2017 ratings, used only for scoring, never for fitting.
- **Catalogue** $\mathcal{C}$: 17,965 movies with at least 10 pre-2017 ratings.
- **Cohort** $\mathcal{U}$: 4,778 users with at least one eligible 2017 positive (169,368 positives).
- **Positive**: an observed rating strictly greater than 3.
- **Candidates** for user $u$: $\mathcal{C}$ minus the movies $u$ rated in $H$.
- **Ranking**: score descending, exact ties broken by ascending `movieId`, top 10.

Eligibility uses only pre-cutoff information, so no future outcome leaks into training or cohort construction.

In [2]:
from src.ratings import load_ratings
from src.splits import VALIDATION_START, TEST_START
from src.pipeline import EvaluationCohort, score_grid

catalogue = pd.read_csv('data/processed/eligible_movies.csv')
positives = pd.read_csv('data/processed/validation_positives.csv')

ratings = load_ratings()
history = ratings.loc[
    ratings['timestamp'] < VALIDATION_START, ['userId', 'movieId', 'rating']
].copy()

# Shared rating-prediction pairs for RMSE/MAE (cohort users x catalogue movies).
outcomes_2017 = ratings.loc[
    (ratings['timestamp'] >= VALIDATION_START) & (ratings['timestamp'] < TEST_START),
    ['userId', 'movieId', 'rating'],
]
del ratings
rating_pairs = (
    outcomes_2017.loc[
        outcomes_2017['userId'].isin(positives['userId'].unique())
        & outcomes_2017['movieId'].isin(catalogue['movieId'])
    ]
    .drop_duplicates(['userId', 'movieId'])
    .reset_index(drop=True)
)

cohort = EvaluationCohort(catalogue, positives)
cohort.set_history(history)

print(f'History rows     : {len(history):,}')
print(f'Catalogue movies : {len(catalogue):,}')
print(f'Evaluated users  : {len(cohort.users):,}')
print(f'Positives        : {len(positives):,}')
print(f'Rating pairs     : {len(rating_pairs):,}')

History rows     : 20,798,765
Catalogue movies : 17,965
Evaluated users  : 4,778
Positives        : 169,368
Rating pairs     : 275,061


## 2. P2.1 — Popularity baseline (ranking)

**Idea.** Recommend what the crowd watched most.

**Score** (rating count in the history, independent of star value):

$$\text{pop}(i) = \sum_{(u,i) \in H} 1$$

The score is the same for every user; personalization comes only from excluding each user's own history. Popularity yields ranking scores, not star predictions, so RMSE and MAE do not apply.

In [3]:
from src.popularity import PopularityModel

def evaluate_model(model, cohort, fallback=None):
    """Fit, score the candidate grid, rank, and summarise."""
    t0 = perf_counter()
    model = model.fit(history)
    fit_seconds = perf_counter() - t0

    t0 = perf_counter()
    scores, missing = score_grid(model, cohort, fallback=fallback)
    score_seconds = perf_counter() - t0

    ranked = cohort.rank_top_k(scores)
    summary, user_metrics = cohort.summary(ranked)
    return {
        'name': model.name,
        'model_obj': model,
        'summary': summary,
        'user_metrics': user_metrics,
        'ranked': ranked,
        'missing_scores': missing,
        'fit_seconds': fit_seconds,
        'score_seconds': score_seconds,
    }

pop_result = evaluate_model(PopularityModel(), cohort)
pd.Series(pop_result['summary'])

users                    4778.000000
mean_ndcg_at_10             0.075212
mean_recall_at_10           0.023637
coverage_movies           287.000000
coverage_fraction           0.015976
less_popular_exposure       0.000000
dtype: float64

## 3. P2.2 — Regularized bias baseline (rating)

**Model.**

$$\hat r_{ui} = \mu + b_u + b_i$$

**Objective** (regularized least squares over observed ratings):

$$\min_{b_u, b_i} \sum_{(u,i) \in H} (r_{ui} - \mu - b_u - b_i)^2 + \lambda_u \sum_u b_u^2 + \lambda_i \sum_i b_i^2$$

**Alternating updates** (closed form when the other side is fixed):

$$b_u \leftarrow \frac{\sum_{i \in I_u} (r_{ui} - \mu - b_i)}{|I_u| + \lambda_u}, \qquad b_i \leftarrow \frac{\sum_{u \in U_i} (r_{ui} - \mu - b_u)}{|U_i| + \lambda_i}$$

The terms $|I_u| + \lambda_u$ shrink noisy biases toward zero, which matters most for users and movies with few ratings. A missing user or movie gets bias 0, which is what makes the bias model the natural fallback for unsupported predictions. Ranking uses raw unclipped predictions; `predict` clips to [1, 5] only for RMSE and MAE.

In [4]:
from src.bias import BiasModel

bias_result = evaluate_model(
    BiasModel(lambda_user=20.0, lambda_movie=20.0, max_passes=20), cohort
)
pd.Series(bias_result['summary'])

users                    4778.000000
mean_ndcg_at_10             0.038258
mean_recall_at_10           0.013524
coverage_movies            81.000000
coverage_fraction           0.004509
less_popular_exposure       0.315404
dtype: float64

## 4. P2.3 — Item-based CF (rating)

**Idea.** Movies liked by the same people are similar; predict a user's rating from how they rated similar movies.

**Adjusted-cosine similarity** (subtract each user mean $\bar r_u$):

$$s_{ij} = \frac{\sum_{u \in U_{ij}} (r_{ui} - \bar r_u)(r_{uj} - \bar r_u)}{\sqrt{\sum_u (r_{ui} - \bar r_u)^2} \sqrt{\sum_u (r_{uj} - \bar r_u)^2}}$$

Keep only positive similarities, then each item keeps its top-$K$ neighbours.

**Prediction** blends the candidate item mean with similarity-weighted residuals:

$$\hat r_{ui} = \bar r_i + \frac{\sum_{j \in N_K(i)} s_{ij} (r_{uj} - \bar r_j)}{\sum_{j \in N_K(i)} |s_{ij}|}$$

If the user has no positively-similar rated item, the score is `NaN` and the shared fallback (the bias model) fills it. Using residuals keeps scores graded instead of saturating at the star ceiling, which would flatten the ranking.

In [5]:
from src.item_cf import ItemCFModel

fallback = BiasModel(lambda_user=20.0, lambda_movie=20.0, max_passes=20).fit(history)
item_cf_result = evaluate_model(
    ItemCFModel(top_k=50, min_similarity=0.0), cohort, fallback=fallback
)
missing = item_cf_result['missing_scores']
print('fallback-filled cells:', f'{missing:,}')
pd.Series(item_cf_result['summary'])

fallback-filled cells: 72,168,530


users                    4778.000000
mean_ndcg_at_10             0.018596
mean_recall_at_10           0.007146
coverage_movies          7467.000000
coverage_fraction           0.415642
less_popular_exposure       0.481750
dtype: float64

## 5. P2.4 — Matrix factorization (explicit ALS with biases)

**Model.**

$$\hat r_{ui} = \mu + b_u + b_i + \mathbf{p}_u^\top \mathbf{q}_i$$

**Objective** over observed ratings with L2 regularization:

$$\min_{P, Q, b} \sum_{(u,i) \in H} \big(r_{ui} - \mu - b_u - b_i - \mathbf{p}_u^\top \mathbf{q}_i\big)^2 + \lambda \big(\lVert \mathbf{p}_u \rVert^2 + \lVert \mathbf{q}_i \rVert^2\big) + \lambda_u b_u^2 + \lambda_i b_i^2$$

**Alternating least squares.** With one side fixed, each row is a ridge regression with a closed form. For user $u$ with rated items $I_u$:

$$\mathbf{p}_u = \Big( \sum_{i \in I_u} \mathbf{q}_i \mathbf{q}_i^\top + \lambda I \Big)^{-1} \sum_{i \in I_u} \mathbf{q}_i (r_{ui} - \mu - b_u - b_i)$$

and symmetrically for $\mathbf{q}_i$. Each pass updates $b_u$, then $b_i$, then $P$, then $Q$ (Gauss–Seidel style); convergence is tracked by training RMSE with a tolerance stop.

**Regularization matters:** a weak $\lambda$ overfits. The validation-selected config is $k = 16$, $\lambda = 15$.

In [6]:
from src.matrix_factorization import MatrixFactorizationModel

mf_result = evaluate_model(
    MatrixFactorizationModel(
        n_factors=16,
        regularization=15.0,
        lambda_user=10.0,
        lambda_movie=10.0,
        max_iter=20,
    ),
    cohort,
)
pd.Series(mf_result['summary'])

users                    4778.000000
mean_ndcg_at_10             0.095399
mean_recall_at_10           0.040035
coverage_movies          1506.000000
coverage_fraction           0.083830
less_popular_exposure       0.219192
dtype: float64

## 6. P2.6 — Shared recommendation lists

Every model goes through the same pipeline (`src/pipeline.py`):

1. `score_candidates(users, movies)` returns a dense score grid.
2. Each user's already-rated movies are set to $-\infty$.
3. Scores are negated and a **stable** top-10 argsort is taken. Because the catalogue is stored in ascending `movieId` order, stable sorting breaks exact ties by ascending `movieId` (the Phase 1 rule).
4. Non-finite scores are replaced by the bias fallback via `score_grid`, and the count is reported.

This guarantees identical eligibility, exclusion and ranking across models, which is a hard Phase 2 acceptance criterion.

In [7]:
results = {
    'popularity': pop_result,
    'bias': bias_result,
    'item_cf': item_cf_result,
    'matrix_factorization': mf_result,
}

ranking_rows = []
for key, res in results.items():
    s = res['summary']
    ranking_rows.append({
        'model': key,
        'mean_ndcg_at_10': s['mean_ndcg_at_10'],
        'mean_recall_at_10': s['mean_recall_at_10'],
        'coverage_movies': s['coverage_movies'],
        'coverage_fraction': s['coverage_fraction'],
        'less_popular_exposure': s['less_popular_exposure'],
        'missing_scores': res['missing_scores'],
    })
ranking_table = pd.DataFrame(ranking_rows).set_index('model')
ranking_table

,mean_ndcg_at_10,mean_recall_at_10,coverage_movies,coverage_fraction,less_popular_exposure,missing_scores
model,,,,,,
popularity,0.075212,0.023637,287,0.015976,0.000000,0
bias,0.038258,0.013524,81,0.004509,0.315404,0
item_cf,0.018596,0.007146,7467,0.415642,0.481750,72168530
matrix_factorization,0.095399,0.040035,1506,0.083830,0.219192,0


## 7. P2.7 — Metrics

**Ranking metrics** (per user with at least one eligible positive; binary relevance):

$$\text{DCG@10} = \sum_{k=1}^{10} \frac{\mathbb{1}[y_k \in \text{positives}]}{\log_2 (k+1)}, \qquad \text{NDCG@10} = \frac{\text{DCG@10}}{\text{IDCG@10}}$$

$$\text{Recall@10} = \frac{|\text{top-10} \cap \text{positives}|}{|\text{positives}|}$$

**Catalogue coverage** at 10 = distinct recommended titles divided by $|\mathcal{C}|$.

**Less-popular exposure** = share of returned slots outside the top fifth of movies by pre-2017 count (that top fifth holds about 89.87 percent of historical ratings).

**Rating metrics** on aligned observed pairs:

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (r_i - \hat r_i)^2}, \qquad \text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |r_i - \hat r_i|$$

In [8]:
from src.metrics import rmse, mae

actual = rating_pairs['rating'].to_numpy(dtype=np.float64)
rating_rows = []
for key, res in results.items():
    model = res['model_obj']
    try:
        predicted = model.predict(rating_pairs)
    except NotImplementedError:
        continue  # popularity has no star predictions
    if not np.isfinite(predicted).all():
        predicted = np.where(
            np.isfinite(predicted), predicted, fallback.predict(rating_pairs)
        )
    rating_rows.append({
        'model': key,
        'rmse': rmse(actual, predicted),
        'mae': mae(actual, predicted),
    })
rating_table = pd.DataFrame(rating_rows).set_index('model')
rating_table

,rmse,mae
model,,
bias,0.851667,0.634622
item_cf,0.870230,0.636891
matrix_factorization,0.798779,0.589331


## 8. P2.8 — Resource requirements

Record fit time, score time and resident memory per model, with the hardware stated so numbers are comparable.

$$\text{RSS (MB)} = \frac{\text{resident bytes}}{1024^2}$$

The cell below reports the current process RSS after the runs. The full runner samples memory around each fit and writes per-model peaks to `outputs/phase2_validation_environment.json`.

In [9]:
resource_rows = []
for key, res in results.items():
    resource_rows.append({
        'model': key,
        'fit_seconds': round(res['fit_seconds'], 2),
        'score_seconds': round(res['score_seconds'], 3),
        'process_rss_mb': round(rss_mb(), 1),
    })
resource_table = pd.DataFrame(resource_rows).set_index('model')
print('Logical CPUs:', psutil.cpu_count(logical=True))
print('Total RAM MB:', round(psutil.virtual_memory().total / 1024 ** 2))
resource_table

Logical CPUs: 16
Total RAM MB: 32118


,fit_seconds,score_seconds,process_rss_mb
model,,,
popularity,0.08,0.193,781.1
bias,7.77,0.336,781.1
item_cf,90.80,3.483,781.1
matrix_factorization,201.40,0.811,781.1


## 9. P2.5 — Tuning budgets

Use a comparable search: three settings per personalized model, selected on 2017 validation by mean NDCG@10 (ties broken by coverage). No test data is used.

- Bias: $\lambda \in \{5, 10, 20\}$
- Item-CF: $K \in \{50, 100, 200\}$
- MF: $(k, \lambda) \in \{(16, 15), (24, 15), (24, 10)\}$

This mirrors `src/tune_models.py`. The live grid is optional because it refits several models.

In [10]:
GRIDS = {
    'bias': [{'lambda_user': v, 'lambda_movie': v} for v in (5.0, 10.0, 20.0)],
    'item_cf': [{'top_k': k, 'min_similarity': 0.0} for k in (50, 100, 200)],
    'matrix_factorization': [
        {'n_factors': k, 'regularization': r}
        for k, r in ((16, 15.0), (24, 15.0), (24, 10.0))
    ],
}
FACTORIES = {
    'bias': BiasModel,
    'item_cf': ItemCFModel,
    'matrix_factorization': MatrixFactorizationModel,
}

RUN_LIVE_GRID = False  # set True to refit every grid entry (slow)
grid_rows = []
if RUN_LIVE_GRID:
    for mname, grid in GRIDS.items():
        for params in grid:
            res = evaluate_model(FACTORIES[mname](**params), cohort, fallback=fallback)
            s = res['summary']
            grid_rows.append({
                'model': mname,
                'params': str(params),
                'mean_ndcg_at_10': s['mean_ndcg_at_10'],
                'coverage_movies': s['coverage_movies'],
            })
    print('Live grid complete.')
else:
    print('Live grid skipped. Set RUN_LIVE_GRID = True to run it.')
    print('Full recorded grid: python -m src.tune_models')

pd.DataFrame(grid_rows)

Live grid skipped. Set RUN_LIVE_GRID = True to run it.
Full recorded grid: python -m src.tune_models


""


## 10. P2.9 — Self-checks and reproduction

Deterministic synthetic tests validate every model and the exclusion rule without the full archive (`src/check_phase2.py`):

- popularity ranks and refuses star predictions,
- bias beats the global-mean RMSE,
- item-CF similarity on co-liked items and `NaN` for unknown users,
- MF recovers low-rank synthetic data,
- the pipeline excludes already-rated movies.

In [11]:
from src import check_phase2

check_phase2.main()

popularity pass
bias pass
item_cf pass
matrix_factorization pass
pipeline exclusion pass
P2 self-checks passed


## 11. Results summary

Combine the ranking and rating tables. Reference values with the P2.5-selected configs: popularity 0.0752, bias 0.0383, Item-CF 0.0186, MF 0.0954 mean NDCG@10, with MF the lowest RMSE.

In [12]:
summary_table = ranking_table.copy()
if len(rating_table):
    summary_table = summary_table.join(rating_table, how='left')
summary_table.round(6)

,mean_ndcg_at_10,mean_recall_at_10,coverage_movies,coverage_fraction,less_popular_exposure,missing_scores,rmse,mae
model,,,,,,,,
popularity,0.075212,0.023637,287,0.015976,0.000000,0,NaN,NaN
bias,0.038258,0.013524,81,0.004509,0.315404,0,0.851667,0.634622
item_cf,0.018596,0.007146,7467,0.415642,0.481750,72168530,0.870230,0.636891
matrix_factorization,0.095399,0.040035,1506,0.083830,0.219192,0,0.798779,0.589331


## 12. Notes and next steps

- Full reproducible run: `python -m src.run_phase2_validation` writes `outputs/phase2_validation_summary.csv`, per-model user metrics, top-10 lists and `phase2_validation_environment.json`.
- Tuning record: `python -m src.tune_models` writes `outputs/phase2_tuning_grid.csv` and `outputs/phase2_selected_config.json`.
- Findings: MF is best on both ranking and rating accuracy. Bias and Item-CF rank below popularity despite lower RMSE (the rating-versus-ranking gap), which motivates the Phase 3 hybrid.
- Remaining: P2.9 independent reproduction and sign-off.